In [1]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.5/95.5 kB 5.4 MB/s eta 0:00:00


In [2]:

from pathlib import Path

import yaml
from ultralytics import YOLO


# CONFIG — edit these values for your setup

ORIGINAL_DATA_YAML = "/kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/data.yaml"


WRITABLE_DIR = "/kaggle/working"   

MODEL_SIZE  = "yolov9m.pt"         
EPOCHS      = 80
IMG_SIZE    = 640
BATCH       = 16
DEVICE      = 0                    
PATIENCE    = 30                   


def build_resolved_data_yaml():
    """
    Read the ORIGINAL data.yaml (whatever relative paths it contains) and
    write a NEW data.yaml with fully absolute train/val/test paths, based on
    where the images actually sit on disk (siblings of the original yaml —
    i.e. <dataset_root>/train/images, <dataset_root>/valid/images,
    <dataset_root>/test/images).
    """
    orig_path = Path(ORIGINAL_DATA_YAML).resolve()
    with open(orig_path) as f:
        cfg = yaml.safe_load(f)

    dataset_root = orig_path.parent  # folder that actually contains train/valid/test

    resolved = {
        "train": str((dataset_root / "train" / "images").resolve()),
        "val":   str((dataset_root / "valid" / "images").resolve()),
        "test":  str((dataset_root / "test" / "images").resolve()),
        "nc":    cfg["nc"],
        "names": cfg["names"],
    }

    for split, img_dir in [("train", resolved["train"]), ("val", resolved["val"]), ("test", resolved["test"])]:
        if not Path(img_dir).exists():
            raise FileNotFoundError(f"'{split}' images dir does not exist: {img_dir}")

    Path(WRITABLE_DIR).mkdir(parents=True, exist_ok=True)
    new_yaml_path = Path(WRITABLE_DIR) / "data_resolved.yaml"
    with open(new_yaml_path, "w") as f:
        yaml.safe_dump(resolved, f, sort_keys=False)

    print(f"Resolved data.yaml written to: {new_yaml_path}")
    for k in ("train", "val", "test"):
        print(f"  {k}: {resolved[k]}")

    return str(new_yaml_path)


def print_metrics(metrics, label):
    print(f"\n--- {label} ---")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall:    {metrics.box.mr:.4f}")
    print(f"mAP50:     {metrics.box.map50:.4f}")
    print(f"mAP50-95:  {metrics.box.map:.4f}")


def train(data_yaml):
    """Train YOLO26 on the custom vehicle dataset (uses Ultralytics' default runs/detect/ output folder)."""
    model = YOLO(MODEL_SIZE)  # loads COCO-pretrained weights

    results = model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        patience=PATIENCE,
        plots=True,
    )

    save_dir = results.save_dir  # wherever Ultralytics decided to save this run
    best_weights = str(save_dir / "weights" / "best.pt")
    print(f"\nTraining complete. Best weights saved to: {best_weights}")
    return best_weights


# ---- run training ----
resolved_yaml = build_resolved_data_yaml()
best_weights = train(resolved_yaml)

# Explicit validation pass on the val split, with precision/recall
val_model = YOLO(best_weights)
val_metrics = val_model.val(data=resolved_yaml, device=DEVICE)
print_metrics(val_metrics, "Validation (val split)")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Resolved data.yaml written to: /kaggle/working/data_resolved.yaml
  train: /kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/train/images
  val: /kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/valid/images
  test: /kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/test/images
Ultralytics 8.4.172 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, 

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/80      7.84G     0.5491     0.9393      1.133         83        640: 100% ━━━━━━━━━━━━ 743/743 1.8it/s 7:04
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 2.1it/s 16.8s
                   all       1136       1366       0.79      0.765      0.849      0.617

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/80      7.76G     0.6041      1.058      1.179         85        640: 100% ━━━━━━━━━━━━ 743/743 1.8it/s 6:54
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 2.1it/s 16.8s
                   all       1136       1366      0.689      0.692      0.733      0.496

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/80      7.83G     0.6045      1.071      1.189         93        640: 100% ━━━━━━━━━━━━ 743/743 1.8it/s 6:50
                 Class     Images  Instances      Box(

In [3]:

import yaml
from ultralytics import YOLO

model = YOLO(best_weights)

# ---- Metrics on the held-out test split ----
test_metrics = model.val(data=resolved_yaml, split="test", device=DEVICE)

print("\nTest split metrics")
print(f"Precision: {test_metrics.box.mp:.4f}")
print(f"Recall:    {test_metrics.box.mr:.4f}")
print(f"mAP50:     {test_metrics.box.map50:.4f}")
print(f"mAP50-95:  {test_metrics.box.map:.4f}")

# Per-class precision/recall/mAP
print("\nPer-class")
names = test_metrics.names
for i, c in enumerate(test_metrics.ap_class_index):
    p, r, ap50, ap = test_metrics.box.class_result(i)
    print(f"{names[c]:<12} P: {p:.3f}  R: {r:.3f}  mAP50: {ap50:.3f}  mAP50-95: {ap:.3f}")

# ---- Save annotated predictions on the test images ----
with open(resolved_yaml) as f:
    cfg = yaml.safe_load(f)
test_images_dir = cfg["test"]

model.predict(
    source=test_images_dir,
    imgsz=IMG_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    verbose=False,
)

print("\nAnnotated test predictions saved (see the 'predict' folder under runs/detect/).")

Ultralytics 8.4.172 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv9m summary (fused): 151 layers, 20,018,053 parameters, 0 gradients, 76.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 7.9±2.1 MB/s, size: 59.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/test/labels... 566 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 566/566 152.6it/s 3.7s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 2.1it/s 17.3s
                   all        566        676      0.974      0.962      0.985      0.844
                   Bus         79         92      0.947  